# 06 — CI/CD for ML

## What
CI/CD for ML extends traditional software CI/CD with three additional concerns: (1) data validation gates, (2) model training and evaluation as part of the pipeline, and (3) model-level acceptance tests rather than just unit tests.

## Why
Code that trains a worse model should fail CI just like code that breaks a unit test. Without automated evaluation gates, regressions silently reach production. GitHub Actions, GitLab CI, and Jenkins are commonly used; the model training step often invokes a cloud training job or Kubernetes batch workload.

## Community context
Google's MLOps whitepaper describes 'continuous training' as a separate axis from continuous integration and delivery — automated retraining triggered by data drift, scheduled cadence, or explicit triggers. The key insight is that model quality degrades over time even when code doesn't change, so training must be part of the automated pipeline.

## GitHub Actions ML CI Pipeline

The following cell shows a realistic GitHub Actions workflow YAML for an ML CI pipeline. This runs on every pull request that touches training code, data configs, or model configs.

```yaml
name: ML CI Pipeline

on:
  push:
    paths:
      - 'src/train/**'
      - 'src/features/**'
      - 'data/configs/**'
      - 'params.yaml'
  pull_request:
    branches: [main]

jobs:
  data-validation:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Setup Python
        uses: actions/setup-python@v5
        with: {python-version: '3.11'}
      - name: Install deps
        run: pip install -r requirements.txt
      - name: Validate data schema
        run: python src/validate_data.py --config data/configs/schema.yaml
      - name: Check data drift vs baseline
        run: python src/drift_check.py --baseline data/baselines/train_stats.json

  train-and-evaluate:
    needs: data-validation
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Setup DVC
        run: pip install dvc[s3]
      - name: Pull data
        run: dvc pull data/train.csv.dvc
      - name: Train model
        run: python src/train.py --params params.yaml --output models/ci_model.pkl
      - name: Evaluate model
        run: python src/evaluate.py --model models/ci_model.pkl --out metrics/ci_eval.json
      - name: Check acceptance thresholds
        run: python src/check_thresholds.py --metrics metrics/ci_eval.json

  model-tests:
    needs: train-and-evaluate
    runs-on: ubuntu-latest
    steps:
      - name: Run model unit tests
        run: pytest tests/model/ -v
      - name: Run fairness checks
        run: python src/fairness_check.py --model models/ci_model.pkl
      - name: Run adversarial robustness tests
        run: python src/robustness_tests.py --model models/ci_model.pkl

  register-model:
    needs: model-tests
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - name: Register to staging
        run: python src/register_model.py --stage Staging
```

In [ ]:
# Simulate the acceptance threshold checker — the key gate in ML CI
import json

THRESHOLDS = {
    'val_auc': {'min': 0.80, 'max_regression_vs_prod': 0.02},
    'val_precision': {'min': 0.75},
    'val_recall': {'min': 0.70},
    'inference_latency_p99_ms': {'max': 100},
    'model_size_mb': {'max': 50},
}

def check_thresholds(ci_metrics, prod_metrics=None):
    failures = []
    for metric, rules in THRESHOLDS.items():
        val = ci_metrics.get(metric)
        if val is None:
            failures.append(f'MISSING metric: {metric}')
            continue
        if 'min' in rules and val < rules['min']:
            failures.append(f'{metric}={val:.3f} < min={rules["min"]}')
        if 'max' in rules and val > rules['max']:
            failures.append(f'{metric}={val:.3f} > max={rules["max"]}')
        if 'max_regression_vs_prod' in rules and prod_metrics and metric in prod_metrics:
            regression = prod_metrics[metric] - val
            if regression > rules['max_regression_vs_prod']:
                failures.append(f'{metric} regressed by {regression:.3f} vs prod ({rules["max_regression_vs_prod"]} allowed)')
    if failures:
        print('CI FAILED - Acceptance threshold violations:')
        for f in failures:
            print(f'  FAIL: {f}')
        return False
    else:
        print('CI PASSED - All acceptance thresholds met')
        return True

# Passing scenario
ci_metrics = {
    'val_auc': 0.85,
    'val_precision': 0.81,
    'val_recall': 0.77,
    'inference_latency_p99_ms': 45,
    'model_size_mb': 12.3,
}
prod_metrics = {'val_auc': 0.84}
print('=== Scenario 1: Good model ===')
check_thresholds(ci_metrics, prod_metrics)

# Failing scenario: AUC regression
ci_bad = {**ci_metrics, 'val_auc': 0.79, 'val_recall': 0.65}
print('\n=== Scenario 2: Regressed model ===')
check_thresholds(ci_bad, prod_metrics)